# New Demands on the IISG Aquifer System

## Overview

This notebook picks up where `intro_base_model.ipynb` left off. We now **modify** the IISG base groundwater model to simulate the impact of a new large-scale pumping demand and explore different strategies for meeting it while minimizing environmental and water-use conflicts.

### Scenario

A township within our model domain must immediately accommodate **10 million gallons per day (MGD)** of new water demand. Using the base model as a starting point, we will:

1. Interactively select a township from the model domain
2. Use geology (transmissivity) to inform where new production wells should be placed
3. Choose monitoring wells to track the aquifer response
4. Run a transient simulation and visualize head and drawdown results

### What you will learn

1. How to spatially join a GIS shapefile to a MODFLOW 6 model grid
2. How to use **transmissivity** as a decision-making tool for well siting
3. How to place new **pumping wells** and **observation points** interactively using an ipyleaflet map
4. How to convert a steady-state simulation to a **transient** simulation and add new wells
5. How to visualize **drawdown** and **head contours** in an interactive dashboard

### Workshop Context

This is one of three scenario notebooks that build on `intro_base_model.ipynb`:

| Notebook | Scenario |
|---|---|
| **-- `new_demands.ipynb`** | **New large-scale pumping demand (this notebook) --** |
| `drought_impacts.ipynb` | Reduced recharge representing a drought |
| `gw_age.ipynb` | Groundwater age and transport modeling |

Each scenario uses the same base model but explores a different type of stress on the aquifer system.

> **Note — Educational Use Only.** The base model is built from real-world geology and
> hydrodynamic data for Chicago's South Suburbs, but simplifying assumptions have been made
> to reduce its size and runtime. Results should **not** be used for planning or decision-making;
> they are intended for exploration and education.

## Imports

The same core libraries used in `intro_base_model.ipynb` are needed here, along with additional packages for interactive web mapping and GIS analysis:

| Library | Purpose |
|---|---|
| `flopy` | Load and modify MODFLOW 6 simulations |
| `numpy`, `pandas` | Array and tabular data operations |
| `matplotlib` | Static plots and colormaps |
| `geopandas`, `shapely` | Spatial operations: shapefile reading, polygon geometry |
| `ipyleaflet` | Interactive web maps in Jupyter |
| `ipywidgets` | UI controls (buttons, toggles, labels) |
| `json` | Serialize GeoDataFrames to GeoJSON for ipyleaflet |

In [ ]:
# Standard Library Imports
from pathlib import Path
import json

# Scientific Stack
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Domain-Specific Packages (Flopy)
import flopy

### Advanced Packages ###

# Neato toys for Jupyter Notebook
from ipywidgets import Layout, Output, VBox, HBox, ToggleButtons, HTML, Button
from ipyleaflet import Map, GeoData, GeoJSON, basemaps, CircleMarker, LayerGroup, WidgetControl
from IPython.display import display, clear_output

# GIS "aids"
import geopandas as gpd
from shapely.geometry import Polygon, LineString

## Housekeeping

We define the same directory structure and path constants as in `intro_base_model.ipynb`. `updated_model_workspace` is where our modified pumping simulation will be written.

> ⚠️ Make sure the `bin/mf6.exe` executable exists at the path printed below before running the simulation cells.

In [ ]:
# Use pathlib to get this notebook's location
notebook_dir = Path.cwd()

# home_dir = notebook_dir.parents[2]
home_dir = notebook_dir.parent

sim_name = "mfsim"

base_model_workspace = home_dir / "modflow_models" / "uncalibrated_iisg_model"
updated_model_workspace = home_dir / "modflow_models" / "new_demand_model"

exe_name = home_dir / "bin" / "mf6.exe"

# shp_townships_path = home_dir / "data" / "background_shapefiles" / "townships_5070_IISGclip.shp"
shp_townships_path = home_dir / "shapefiles" / "townships_5070_IISGclip.shp"
shp_IISG_model_bounds_path = base_model_workspace / "postproc" / "shps" / "gwf-iisg_bbox.shp"




print("Base model workspace :", base_model_workspace)
print("New demand model workspace:", updated_model_workspace)

## Loading the Base Model

We load the IISG MODFLOW 6 base model using FloPy's `MFSimulation.load()`. This is a **steady-state** simulation that represents long-term equilibrium conditions before any new stresses are applied.

This model will serve as:
- The **source** of initial conditions (steady-state heads) for the transient run
- The **reference** for computing drawdown: `drawdown = pumping heads − base heads`
- The **template** that will be copied and modified for the pumping scenario

In [ ]:
# Load the base simulation
base_sim = flopy.mf6.MFSimulation.load(
    sim_name=sim_name,
    sim_ws=base_model_workspace,
    write_headers=False,
    exe_name=exe_name
)

print("\n-- Success! Base model loaded to new workspace --\n")

## Extracting the Groundwater Flow Model

The loaded simulation may contain multiple models. We extract the groundwater flow (GWF) model object and pre-compute several quantities we will need throughout the notebook:

- **Grid dimensions** (`nrow`, `ncol`, `nlay`)
- **Layer elevations** (tops and bottoms)
- **Horizontal hydraulic conductivity** (`Kh`)
- **Layer thickness** and **simple transmissivity** (`T = Kh × thickness`)

Transmissivity is the geological variable we will use to identify the most productive locations for new wells — higher T means faster groundwater flow and greater well yield.

In [ ]:
# Extract the groundwater flow model
base_model_name = list(base_sim.model_names)[0]
base_gwf = base_sim.get_model(base_model_name)

# Other useful metadata about the model
model_top_elevation = base_gwf.modelgrid.top
layer_botm_elevations = base_gwf.modelgrid.botm

nrow_base = base_gwf.modelgrid.nrow
ncol_base = base_gwf.modelgrid.ncol
nlay_base = base_gwf.modelgrid.nlay

base_kh = base_gwf.npf.k.array

base_thickness = np.zeros_like(base_kh)

for layer in range(nlay_base):
    if layer == 0:
        base_thickness[layer] = model_top_elevation - layer_botm_elevations[layer]
    else:
        base_thickness[layer] = layer_botm_elevations[layer-1] - layer_botm_elevations[layer]

simple_transmissivity = base_kh * base_thickness

print(f"Base model loaded: {base_model_name}")
print(f"Grid: {nlay_base} layers x {nrow_base} rows x {ncol_base} cols")

## Selecting a Township

Our model domain spans multiple townships. The interactive map below lets you **click** on any township to select it as the focus of the analysis.

The township layer is colored using the IISG blue/orange palette:
- **Hovering** over a township highlights it in orange
- **Clicking** locks in your selection (displayed below the map)

> 🖱️ Click a township on the map, then run the next cell to confirm your selection before continuing.

In [ ]:
# Load shapefiles
townships_gdf = gpd.read_file(shp_townships_path)
bounds_gdf = gpd.read_file(shp_IISG_model_bounds_path)

# Web maps require WGS84 (EPSG:4326) coordinate reference system
townships_gdf = townships_gdf.to_crs(epsg=4326)
bounds_gdf = bounds_gdf.to_crs(epsg=4326)

# Calculate center and initialize the map
tb = bounds_gdf.total_bounds 
center_y = (tb[1] + tb[3]) / 2
center_x = (tb[0] + tb[2]) / 2

# Increase the layout height and set an explicit center/zoom
# Adjust the zoom integer (e.g., 9, 10, or 11) to get the perfect fit
m = Map(
    basemap=basemaps.Esri.WorldTopoMap, 
    center=(center_y, center_x), 
    zoom=9, 
    layout=Layout(height='725px')
)

# Create GeoData layer with styling
townships_layer = GeoData(
    geo_dataframe=townships_gdf,
    style={
        'color': '#13294B',
        'fillColor': '#13294B',
        'opacity': 1,
        'weight': 2,
        'fillOpacity': 0.6
    },
    hover_style={
        'fillColor': '#FF5F05',
        'fillOpacity': 0.8
    },
    name='Townships'
)

# Set up interaction state and UI labels
selected_township = None

hover_label = HTML(value="<b>Hovered Township:</b> <i>None</i>")
click_label = HTML(value="<b>Selected Township:</b> <i>None</i>")

def on_hover(event, feature, **kwargs):
    township_name = feature['properties'].get('NAME', 'Unknown')
    hover_label.value = f"<b>Hovered Township:</b> {township_name}"

def on_click(event, feature, **kwargs):
    global selected_township
    selected_township = feature['properties'].get('NAME', 'Unknown')
    click_label.value = f"<b>Selected Township:</b> <span style='color:#FF5F05'>{selected_township}</span>"

townships_layer.on_hover(on_hover)
townships_layer.on_click(on_click)

m.add(townships_layer)

# Display the dashboard elements
dashboard = VBox([hover_label, click_label, m])
display(dashboard)

In [ ]:
# Just confirming it worked.
print(selected_township)

## Mapping Township Boundaries to the Model Grid

Township boundaries are defined in geographic space (a shapefile), but MODFLOW works in row/column grid space. We need to identify which model cells fall inside the selected township.

The `get_township_dict` function below uses a **spatial join** to accomplish this:

1. Extract the (x, y) center coordinates of every model cell
2. Create a GeoDataFrame of cell center points in the model's native CRS (EPSG:5070)
3. Spatially join those points against the township polygons
4. Return a dictionary mapping township names to arrays of `[row, col]` index pairs

This function is defined once and reused for any township in the model domain.

In [ ]:
def get_township_dict(model, shapefile_path, township_col='NAME', sim_name='mfsim'):
    """
    Creates a dictionary mapping township names to a NumPy array of [row, col] 
    0-indexed pairs for a given MODFLOW 6 model and township shapefile.
    
    Parameters:
        model (modflow6 model object): i.e. base_gwf
        shapefile_path (str or Path): Path to the township shapefile.
        township_col (str): The column name in the shapefile containing township names.
        sim_name (str): The simulation name.
        
    Returns:
        dict: A dictionary where keys are township names and values are 
              2D NumPy arrays of [row, col] integers (0-indexed).
    """

    modelgrid = model.modelgrid 

    # Extract 2D arrays of cell center coordinates
    xcenters = modelgrid.xcellcenters
    ycenters = modelgrid.ycellcenters
    nrow = modelgrid.nrow
    ncol = modelgrid.ncol
    print("Extracted 2D arrays of cell center coordinates")

    # Create arrays of row and column indices (0-based indexing)
    rows, cols = np.indices((nrow, ncol))
    rows = rows.flatten()
    cols = cols.flatten()
    print("Created arrays of row and column indices (0-based indexing)")

    # Flatten the coordinate arrays to match the indices
    x_flat = xcenters.flatten()
    y_flat = ycenters.flatten()
    print("Flattened the coordinate arrays to match the indices")

    # Create a GeoDataFrame of the cell centers
    cell_points = gpd.points_from_xy(x_flat, y_flat)
    gdf_cells = gpd.GeoDataFrame({'row': rows, 'col': cols}, geometry=cell_points)
    print("Created a GeoDataFrame of the cell centers")

    # Load the township shapefile
    townships = gpd.read_file(shapefile_path)
    print("Loaded the township shapefile")

    # Ensure the cell points are assigned the exact same CRS as the shapefile
    gdf_cells.crs = townships.crs

    # Perform the spatial join
    # how='inner' drops cells outside the polygons
    joined = gpd.sjoin(gdf_cells, townships, how='inner', predicate='intersects')
    print("Performed spatial join between model cells and township geometries.")

    # Build and return the lookup dictionary
    lookup_dict = {}
    for township_name, group in joined.groupby(township_col):
        # Extract row/col as a 2D NumPy array and ensure integer dtype
        cell_pairs_array = group[['row', 'col']].values.astype(int)
        lookup_dict[township_name] = cell_pairs_array
    print("lookup dictionary complete!")

    return lookup_dict

### Building and Querying the Township Dictionary

We call `get_township_dict` to build the lookup table for all townships, then retrieve the specific row/column pairs for our selected township.

In [ ]:
township_dict = get_township_dict(
    model=base_gwf, 
    shapefile_path=shp_townships_path
)

In [ ]:
# Accessing the NumPy array for a specific township
township_cells = township_dict.get(selected_township)
if township_cells is not None:
    print(f"Shape of {selected_township} array: {township_cells.shape}")
    print(f"Data type: {township_cells.dtype}")
    print(township_dict[selected_township])

## Visualizing Transmissivity in the Selected Township

Before siting new wells, we need to understand the subsurface geology of our township. **Transmissivity (T)** — the product of hydraulic conductivity and saturated thickness — is the most important indicator of how productive a well location will be.

### What to look for
- **High T zones** (blue colors on the `managua` colormap): these are the best candidates for high-yield production wells
- **Red squares**: existing pumping wells (Layer 9, the deep sand-and-gravel aquifer)
- **Green squares**: surface stream reaches (SFR package, Layer 1)

Placing new wells far from existing wells and streams reduces interference and minimizes stream depletion — an important environmental consideration.

The two cells below compute a 2D township mask and then plot the masked transmissivity.

In [ ]:
# Extract rows and columns
rows = township_cells[:, 0]
cols = township_cells[:, 1]

# 1. Create a 2D boolean mask for the township (True means it will be masked/hidden)
nrow, ncol = base_gwf.modelgrid.nrow, base_gwf.modelgrid.ncol
township_mask_2d = np.ones((nrow, ncol), dtype=bool)

# Unmask the cells that belong to the target township
township_mask_2d[rows, cols] = False 

# 2. Extract idomain and broadcast our 2D township mask to 3D
idomain = base_gwf.modelgrid.idomain
township_mask_3d = np.broadcast_to(township_mask_2d, idomain.shape)

# 3. Combine masks: Hide cell if idomain is 0 OR if it falls outside the township
combined_mask = (idomain == 0) | township_mask_3d

# Apply the combined mask to your KH array
masked_T = np.ma.masked_where(combined_mask, simple_transmissivity)

In [ ]:
# --- Plotting ---
with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_aspect('equal')

    # Create Map View from base model
    mapview = flopy.plot.PlotMapView(model=base_gwf, layer=8, ax=ax)
        
    # Plot masked T over the subdomain
    csa = mapview.plot_array(masked_T, ax=ax, cmap="managua", alpha=1.0, 
                                vmin=masked_T.min(), vmax=masked_T.max())
    
    cb = fig.colorbar(csa, ax=ax, shrink=0.6)
    cb.set_label("Transmissivity - T (m2/d)")

    # Plot boundary conditions 
    mapview.plot_bc("WEL", color="red")
    ax.scatter([], [], c="red", marker="s", s=50, label="Pumping Wells (Layer 9)")

    mapview_sfr = flopy.plot.PlotMapView(model=base_gwf, layer=0, ax=ax)
    mapview_sfr.plot_bc("SFR", color="green")
    ax.scatter([], [], c="green", marker="s", s=50, label="Surface Streams (SFR, Layer 1)")
    ax.legend(loc="upper right")

    # 4. Zoom the plot into the township bounds
    # Get cell center coordinates for the target cells
    xcenters = base_gwf.modelgrid.xcellcenters[rows, cols]
    ycenters = base_gwf.modelgrid.ycellcenters[rows, cols]
    
    # Calculate a slight buffer (approx 2 cells wide) so the plot doesn't hug the very edge
    dx_buffer = base_gwf.modelgrid.delr.mean() * 2
    dy_buffer = base_gwf.modelgrid.delc.mean() * 2
    
    ax.set_xlim(xcenters.min() - dx_buffer, xcenters.max() + dx_buffer)
    ax.set_ylim(ycenters.min() - dy_buffer, ycenters.max() + dy_buffer)

    # Final spit and polish
    plt.title(f"Transmissivity - {selected_township} Township - Layer 9")
    fig.tight_layout()

    plt.show()


## Helper Functions for Interactive Maps

The next two sections (well placement and observation point selection) both require:

1. Identifying existing wells and stream reaches in the township
2. Building a GeoDataFrame of cells with geometry, transmissivity, and validity flags
3. Projecting to WGS84 for web mapping
4. Constructing a colormap
5. Initializing an ipyleaflet Map

Rather than repeating all of this code, we define three helper functions and compute the feature masks **once** before any interactive map is shown.

| Function | Purpose |
|---|---|
| `build_feature_masks` | Build boolean arrays for existing wells and stream reaches |
| `build_cells_gdf` | Build a GeoDataFrame of township cells for web mapping |
| `build_township_map` | Initialize a centered ipyleaflet Map |

In [ ]:
def build_feature_masks(model, nrow, ncol, wel_layer=8, sfr_layer=0):
    """
    Build 2D boolean arrays marking existing pumping wells and stream reaches.

    Parameters
    ----------
    model : flopy.mf6.ModflowGwf
    nrow, ncol : int
    wel_layer : int, optional
        Zero-based layer index for pumping wells. Default is 8 (Layer 9,
        the deep sand-and-gravel aquifer in the IISG model).
    sfr_layer : int, optional
        Zero-based layer index for SFR reaches. Default is 0 (Layer 1).

    Returns
    -------
    has_wel : np.ndarray of bool, shape (nrow, ncol)
    has_sfr : np.ndarray of bool, shape (nrow, ncol)
    """
    has_wel = np.zeros((nrow, ncol), dtype=bool)
    if model.wel is not None:
        spd = model.wel.stress_period_data.get_data()[0]
        if spd is not None:
            for rec in spd:
                l, r, c = rec['cellid']
                if l == wel_layer:
                    has_wel[r, c] = True

    has_sfr = np.zeros((nrow, ncol), dtype=bool)
    if model.sfr is not None:
        pd_arr = model.sfr.packagedata.array
        if pd_arr is not None:
            for rec in pd_arr:
                l, r, c = rec['cellid']
                if l == sfr_layer:
                    has_sfr[r, c] = True

    return has_wel, has_sfr


def build_cells_gdf(model, rows, cols, layer, T_array, has_wel, has_sfr, has_new_wel=None):
    """
    Build a GeoDataFrame of model cells inside the selected township.

    Each cell is annotated with transmissivity, boundary-condition flags, a validity
    flag (True = eligible for well/obs placement), and a display color derived from
    the transmissivity value.

    Parameters
    ----------
    model : flopy.mf6.ModflowGwf
    rows, cols : array_like of int
        Zero-based row and column indices of township cells.
    layer : int
        Zero-based model layer index to sample transmissivity from.
    T_array : np.ndarray
        3D transmissivity array (nlay x nrow x ncol).
    has_wel : np.ndarray of bool, shape (nrow, ncol)
        Existing pumping well locations (from `build_feature_masks`).
    has_sfr : np.ndarray of bool, shape (nrow, ncol)
        Stream reach locations (from `build_feature_masks`).
    has_new_wel : np.ndarray of bool or None, optional
        Newly proposed well locations. When provided, these cells are also marked
        invalid for placement. Default is None.

    Returns
    -------
    cells_gdf : geopandas.GeoDataFrame
        GeoDataFrame in EPSG:4326 with columns: row, col, T, is_wel, is_sfr,
        [is_new_wel], is_valid, centroid_x, centroid_y, hex_color.
    cells_geojson : dict
        JSON-serializable GeoJSON representation of cells_gdf.
    """
    polygons, T_values, row_list, col_list, wel_flags, sfr_flags = [], [], [], [], [], []
    new_wel_flags = [] if has_new_wel is not None else None

    for r, c in zip(rows, cols):
        vertices = model.modelgrid.get_cell_vertices(r, c)
        polygons.append(Polygon(vertices))
        T_values.append(T_array[layer, r, c])
        row_list.append(r)
        col_list.append(c)
        wel_flags.append(bool(has_wel[r, c]))
        sfr_flags.append(bool(has_sfr[r, c]))
        if has_new_wel is not None:
            new_wel_flags.append(bool(has_new_wel[r, c]))

    data = {'row': row_list, 'col': col_list, 'T': T_values,
            'is_wel': wel_flags, 'is_sfr': sfr_flags}
    if has_new_wel is not None:
        data['is_new_wel'] = new_wel_flags

    gdf = gpd.GeoDataFrame(data, geometry=polygons, crs='EPSG:5070')

    if has_new_wel is not None:
        gdf['is_valid'] = ~(gdf['is_wel'] | gdf['is_sfr'] | gdf['is_new_wel'] | (gdf['T'] >= 1e20))
    else:
        gdf['is_valid'] = ~(gdf['is_wel'] | gdf['is_sfr'] | (gdf['T'] >= 1e20))

    # Compute centroids before reprojecting to avoid CRS warnings
    centroids = gdf.geometry.centroid
    gdf = gdf.to_crs(epsg=4326)
    centroids_wgs84 = centroids.to_crs(epsg=4326)
    gdf['centroid_x'] = centroids_wgs84.x
    gdf['centroid_y'] = centroids_wgs84.y

    # Build a transmissivity-based colormap
    T_array = np.array(T_values)
    valid_T = T_array[T_array < 1e20]
    vmin_t = valid_T.min() if len(valid_T) > 0 else 0
    vmax_t = valid_T.max() if len(valid_T) > 0 else 1
    norm = mcolors.Normalize(vmin=vmin_t, vmax=vmax_t)
    try:
        cmap_t = mpl.colormaps.get_cmap('managua')
    except ValueError:
        cmap_t = mpl.colormaps.get_cmap('viridis')

    def _get_color(row):
        if row['is_wel']:
            return '#FF0000'  # Existing well: red
        if has_new_wel is not None and row.get('is_new_wel', False):
            return '#780606'  # Proposed well: dark red
        if row['is_sfr']:
            return '#00FF00'  # Stream reach: green
        if row['T'] >= 1e20:
            return '#000000'  # Inactive cell: black
        return mcolors.to_hex(cmap_t(norm(row['T'])))

    gdf['hex_color'] = gdf.apply(_get_color, axis=1)
    return gdf, json.loads(gdf.to_json())


def build_township_map(cells_gdf, zoom=11):
    """
    Initialize an ipyleaflet Map centered on the given township cells GeoDataFrame.

    Parameters
    ----------
    cells_gdf : geopandas.GeoDataFrame
        GeoDataFrame in EPSG:4326.
    zoom : int, optional
        Initial zoom level. Default is 11.

    Returns
    -------
    ipyleaflet.Map
    """
    tb = cells_gdf.total_bounds
    center_y = (tb[1] + tb[3]) / 2
    center_x = (tb[0] + tb[2]) / 2
    return Map(
        basemap=basemaps.Esri.WorldTopoMap,
        center=(center_y, center_x),
        zoom=zoom,
        layout=Layout(height='750px')
    )

In [ ]:
# Compute feature masks once — reused by both the well placement
# and observation point interactive maps below.
layer_for_wells = 8  # Layer 9 (0-indexed): the bottommost sand-and-gravel aquifer

has_wel, has_sfr = build_feature_masks(
    model=base_gwf,
    nrow=nrow_base,
    ncol=ncol_base,
    wel_layer=layer_for_wells,
    sfr_layer=0
)

print(f"Existing modelwide wells in Layer {layer_for_wells + 1}: {has_wel.sum()} cells")
print(f'Modelwide stream reaches in Layer 1: {has_sfr.sum()} cells')

#todo: trim to township domain

## Configuring Your Well Placement Strategy

Before opening the interactive map, choose two parameters:

| Parameter | Options | Description |
|---|---|---|
| `how_many_wells` | 1 – 10 | Number of new production wells to place |
| `placement_strategy` | `'cluster'` or `'diffuse'` | Spatial arrangement philosophy |

**Cluster strategy**: wells are grouped close together — 
> Maximizes short-term yield but concentrates drawdown.

**Diffuse strategy**: wells are spread across the township — 
> Distributes drawdown more evenly, reducing local depletion. However, this    strategy is more expensive (due to pipeline cost), and can potentially increase required demand (due to water loss enroute).

> 💡 The strategy label is informational — the analysis will not actually enforce any restrictions.

In [ ]:
# --- Configuration ---
# No more than 10 please! Wells cost $$$ to drill, install, maintain.
how_many_wells = 5
# This is more of a reminder for ourself. Nothing is enforced r/ the placement strategy
# at this stage... except for choosing one of the possible options.
placement_strategy = "cluster" #cluster or diffuse

assert placement_strategy in ["cluster", "diffuse"], \
    f"Invalid placement_strategy: {placement_strategy}. Expected one of ['cluster', 'diffuse']"

assert how_many_wells < 11, \
    f"No more than 10 wells please! Wells cost $$$ to drill, install, maintain."

## Step 1 — Selecting Well Locations (Interactive Map)

The map below shows every model cell in the selected township, colored by **transmissivity** in Layer 9 (EPSG:4326 web projection):

| Color | Meaning |
|---|---|
| Colormap (managua) | Eligible cells — color encodes transmissivity |
| 🔴 Red | Existing pumping well (ineligible) |
| 🟢 Green | Surface stream reach — SFR (ineligible) |
| ⚫ Black | Inactive model cell (ineligible) |

**How to use:**
1. Hover over any cell to see its layer/row/column and transmissivity
2. Click eligible cells to select them as well locations (orange markers appear)
3. Click again to deselect
4. Once you have selected exactly `how_many_wells` cells, the **Lock In Picks** button activates — press it to save your selection to `new_wells`

> ⚠️ You must lock in your picks before proceeding to the next cell.

In [ ]:
# Build the cells GeoDataFrame for well placement
# (no has_new_wel at this stage — no proposed wells yet)
cells_gdf, cells_geojson = build_cells_gdf(
    model=base_gwf,
    rows=rows,
    cols=cols,
    layer=layer_for_wells,
    T_array=simple_transmissivity,
    has_wel=has_wel,
    has_sfr=has_sfr
)

# Initialize the map
m_grid = build_township_map(cells_gdf)

def style_callback(feature):
    return {
        'fillColor': feature['properties']['hex_color'],
        'color': '#13294B',
        'weight': 0.5,
        'fillOpacity': 0.8
    }

hover_style_dict = {'fillOpacity': 0.9, 'weight': 2}

grid_layer = GeoJSON(
    data=cells_geojson,
    style_callback=style_callback,
    hover_style=hover_style_dict,
    name=f'{selected_township} T Grid'
)

# Create a dedicated layer group for selection markers
selection_group = LayerGroup()
m_grid.add(grid_layer)
m_grid.add(selection_group)

# Interaction state
selected_cells = {}  # (layer, row, col) -> CircleMarker
new_wells = []       # Final export array

hover_label = HTML(value='<b>Hovered Cell:</b> <i>None</i>')
click_label = HTML(value='<b>Selected Wells:</b> <i>None</i>')

lock_button = Button(
    description='Lock In Picks',
    disabled=True,
    button_style='',
    icon='check'
)

def update_ui():
    if len(selected_cells) == 0:
        click_label.value = '<b>Selected Wells:</b> <i>None</i>'
    else:
        picks_text = '<br>'.join([f'Layer {l+1}, Row {r+1}, Col {c+1}' for (l, r, c) in selected_cells.keys()])
        click_label.value = f'<b>Selected Wells:</b><br><span style="color:#FF5F05">{picks_text}</span>'
    lock_button.disabled = len(selected_cells) != how_many_wells
    lock_button.button_style = 'warning' if not lock_button.disabled else ''

def on_hover(event, feature, **kwargs):
    r = feature['properties']['row']
    c = feature['properties']['col']
    T = feature['properties']['T']
    is_wel = feature['properties']['is_wel']
    is_sfr = feature['properties']['is_sfr']
    if is_wel:
        T_display = "<b><span style='color:red'>Existing Well (Layer 9)</span></b>"
    elif is_sfr:
        T_display = "<b><span style='color:green'>Surface Stream (Layer 1)</span></b>"
    else:
        T_display = f'{T:.2f} m^2/d' if T < 1e20 else 'Inactive'
    hover_label.value = f'<b>Hovered Cell:</b> Layer {layer_for_wells+1}, Row {r+1}, Col {c+1} | <b>Transmissivity:</b> {T_display}'

def on_click(event, feature, **kwargs):
    if lock_button.description == 'Locked!':
        return
    if not feature['properties']['is_valid']:
        return
    r = feature['properties']['row']
    c = feature['properties']['col']
    cell_tuple = (layer_for_wells, r, c)
    if cell_tuple in selected_cells:
        selection_group.remove_layer(selected_cells[cell_tuple])
        del selected_cells[cell_tuple]
    else:
        if len(selected_cells) >= how_many_wells:
            return
        lat = feature['properties']['centroid_y']
        lon = feature['properties']['centroid_x']
        marker = CircleMarker(
            location=(lat, lon), radius=7,
            color='#13294B', fillColor='#FF5F05', fillOpacity=1.0, weight=2
        )
        selection_group.add_layer(marker)
        selected_cells[cell_tuple] = marker
    update_ui()

def on_lock_click(b):
    global new_wells
    new_wells = list(selected_cells.keys())
    b.description = 'Locked!'
    b.button_style = 'success'
    b.disabled = True
    click_label.value += "<br><br><b><span style='color:green'>Selections Locked! Array saved to 'new_wells'.</span></b>"

grid_layer.on_hover(on_hover)
grid_layer.on_click(on_click)
lock_button.on_click(on_lock_click)

strategy_text = f' ({placement_strategy.title()} Strategy)' if how_many_wells > 1 else ''
header = HTML(value=f'<h2>Well Placement — {selected_township} Township</h2>'
             f'<b>Target:</b> {how_many_wells} wells{strategy_text}')

dashboard = VBox([header, hover_label, HBox([click_label, lock_button]), m_grid])
display(dashboard)

### Confirming Well Selections

Run the cell below to confirm that `new_wells` was populated correctly. Each entry is a `(layer, row, col)` tuple (0-indexed).

In [ ]:
print(new_wells)

## Step 2 — Selecting Observation Points (Interactive Map)

Observation points (monitoring wells) allow us to track head changes over time at specific locations. Good placement strategies include:

- **Near the new production wells** — to capture the primary drawdown cone
- **Downgradient of the pumping zone** — to detect head recovery or delayed response
- **Near stream features** — to monitor potential stream depletion

The map below is similar to the well placement map but now also marks the proposed production wells in **dark red** (ineligible for observation points). A minimum of 1 and a maximum of `max_obs_points` can be selected.

**How to use:**
1. Select up to `max_obs_points` cells as monitoring locations (navy markers)
2. Press **Lock In Picks** to save your selection to `obs_points`

> ⚠️ You must lock in your observation points before proceeding.

In [ ]:
# --- Configuration ---
max_obs_points = 10

# Build a has_new_wel mask from the locked-in production well selections
has_new_wel = np.zeros((nrow_base, ncol_base), dtype=bool)
if 'new_wells' in globals():
    for (l, r, c) in new_wells:
        if l == layer_for_wells:
            has_new_wel[r, c] = True

# Build the cells GeoDataFrame — this time including has_new_wel
# so proposed wells are excluded from valid observation locations
cells_gdf, cells_geojson = build_cells_gdf(
    model=base_gwf,
    rows=rows,
    cols=cols,
    layer=layer_for_wells,
    T_array=simple_transmissivity,
    has_wel=has_wel,
    has_sfr=has_sfr,
    has_new_wel=has_new_wel
)

# Initialize the map
m_grid = build_township_map(cells_gdf)

def style_callback(feature):
    return {
        'fillColor': feature['properties']['hex_color'],
        'color': '#13294B',
        'weight': 0.5,
        'fillOpacity': 0.8
    }

hover_style_dict = {'fillOpacity': 0.9, 'weight': 2}

grid_layer = GeoJSON(
    data=cells_geojson,
    style_callback=style_callback,
    hover_style=hover_style_dict,
    name=f'{selected_township} T Grid'
)

selection_group = LayerGroup()
m_grid.add(grid_layer)
m_grid.add(selection_group)

# Interaction state
selected_cells = {}
obs_points = []  # Final export array

hover_label = HTML(value='<b>Hovered Cell:</b> <i>None</i>')
click_label = HTML(value='<b>Selected Obs Points:</b> <i>None</i>')

lock_button = Button(
    description='Lock In Picks',
    disabled=True,
    button_style='',
    icon='check'
)

def update_ui():
    if len(selected_cells) == 0:
        click_label.value = '<b>Selected Obs Points:</b> <i>None</i>'
        lock_button.disabled = True
        lock_button.button_style = ''
    else:
        picks_text = '<br>'.join([f'Layer {l+1}, Row {r+1}, Col {c+1}' for (l, r, c) in selected_cells.keys()])
        click_label.value = f'<b>Selected Obs Points ({len(selected_cells)}/{max_obs_points}):</b><br><span style="color:#13294B">{picks_text}</span>'
        lock_button.disabled = False
        lock_button.button_style = 'warning'

def on_hover(event, feature, **kwargs):
    r = feature['properties']['row']
    c = feature['properties']['col']
    T = feature['properties']['T']
    is_wel = feature['properties']['is_wel']
    is_new_wel = feature['properties'].get('is_new_wel', False)
    is_sfr = feature['properties']['is_sfr']
    if is_wel:
        T_display = "<b><span style='color:red'>Existing Well (Layer 9)</span></b>"
    elif is_new_wel:
        T_display = "<b><span style='color:red'>Proposed Well (Layer 9)</span></b>"
    elif is_sfr:
        T_display = "<b><span style='color:green'>Surface Stream (Layer 1)</span></b>"
    else:
        T_display = f'{T:.2f} m^2/d' if T < 1e20 else 'Inactive'
    hover_label.value = f'<b>Hovered Cell:</b> Layer {layer_for_wells+1}, Row {r+1}, Col {c+1} | <b>Feature/Transmissivity:</b> {T_display}'

def on_click(event, feature, **kwargs):
    if lock_button.description == 'Locked!':
        return
    if not feature['properties']['is_valid']:
        return
    r = feature['properties']['row']
    c = feature['properties']['col']
    cell_tuple = (layer_for_wells, r, c)
    if cell_tuple in selected_cells:
        selection_group.remove_layer(selected_cells[cell_tuple])
        del selected_cells[cell_tuple]
    else:
        if len(selected_cells) >= max_obs_points:
            return
        lat = feature['properties']['centroid_y']
        lon = feature['properties']['centroid_x']
        marker = CircleMarker(
            location=(lat, lon), radius=7,
            color='#FF5F05', fillColor='#13294B', fillOpacity=1.0, weight=2
        )
        selection_group.add_layer(marker)
        selected_cells[cell_tuple] = marker
    update_ui()

def on_lock_click(b):
    global obs_points
    obs_points = list(selected_cells.keys())
    b.description = 'Locked!'
    b.button_style = 'success'
    b.disabled = True
    click_label.value += "<br><br><b><span style='color:green'>Selections Locked! Array saved to 'obs_points'.</span></b>"

grid_layer.on_hover(on_hover)
grid_layer.on_click(on_click)
lock_button.on_click(on_lock_click)

header = HTML(value=f'<h2>Observation Points — {selected_township} Township</h2>'
             f'<b>Target:</b> Select up to {max_obs_points} cells for observation points')

dashboard = VBox([header, hover_label, HBox([click_label, lock_button]), m_grid])
display(dashboard)

### Confirming Observation Point Selections

Run the cell below to confirm that `obs_points` was populated correctly.

In [ ]:
print(obs_points)

## Copying the Base Model to a New Workspace

To avoid modifying the original base model files, we create a copy of the simulation in a new directory (`new_demand_model/`). The same two-step trick used in `drought_impacts.ipynb` is applied here:

1. **Temporarily** redirect the simulation's internal path to `updated_model_workspace`
2. Call `write_simulation()` — FloPy writes all input files to the new location
3. **Restore** the original path so `base_sim` still points to the base model

After copying, we load the copy as `pump_sim` to begin modifying it.


In [ ]:
# "Copy" the base model to the new model workspace by temporarily updating the path
# and calling the write_simulation method.
# There are more elegant (and perhaps more correct) ways to accomplish, but this method only
# requires 2 lines of code :)
base_sim.set_sim_path(updated_model_workspace)
base_sim.write_simulation()

# Undo the horrible thing that was done
base_sim.set_sim_path(base_model_workspace)

print("\n-- Success! Base model copied to new workspace --\n")

In [ ]:
# Load the newly created pumping simulation
pump_sim = flopy.mf6.MFSimulation.load(
    sim_name=sim_name,
    sim_ws=updated_model_workspace,
    write_headers=False,
    exe_name=exe_name
)

print("\n-- Success! Pumping simulation loaded --\n")

## Extracting the Pump Model

Just as with the base model, we extract the GWF model object from the newly loaded pumping simulation. All subsequent package modifications will be applied to `pump_gwf`.

In [ ]:
# Just like with the base model let's extract the groundwater flow model

pump_model_name = list(pump_sim.model_names)[0]

print(pump_model_name)

pump_gwf = pump_sim.get_model(pump_model_name)

print(Path(pump_gwf.model_ws) == updated_model_workspace)

## Configuring the Pumping Scenario

We now set the pumping parameters:

| Parameter | Value | Description |
|---|---|---|
| `pumping_rate_mgd` | 10.0 MGD | Total demand to be met by the new well field |
| `pump_length` | 365.25 days | Duration of the pumping stress period (one year) |

The total demand is converted from **million gallons per day (MGD)** to **m³/day** (the model's native unit) using the conversion factor 1 MGD = 3785.41 m³/day, then divided equally among the selected production wells.

For example, 10 MGD across 3 wells = ~12,618 m³/day per well.

In [ ]:
# --- Distribute Pumping Rates ---
pumping_rate_mgd = 10.0 # mgd
pump_length = 365.25 # days

# Convert to m3/day and divide evenly among the selected wells
total_pumping_m3d = pumping_rate_mgd * 3785.411784
rate_per_well = total_pumping_m3d / len(new_wells)

print(f"Per well pumping rate: {rate_per_well :.2f} m3/day")

## Converting from Steady-State to Transient

The base model is **steady-state** — it has no time dimension. To simulate how heads change over the course of a year of pumping, we must convert it to a **transient** simulation.

### MODFLOW Time Discretization (TDIS) Package

The TDIS package controls:
- The number of **stress periods** (blocks of time with constant boundary conditions)
- The **length** of each stress period (days)
- The **number of time steps** within each period
- The **time-step multiplier** (allows non-uniform stepping within a period)

We use a single one-year stress period with a time-step multiplier of 1.5 (geometric growth), which gives finer resolution early in the pumping period when head changes are most rapid. We first inspect the existing TDIS data, then replace it.

In [ ]:
pump_time_dis = pump_sim.get_package("tdis")

print(pump_time_dis.perioddata.get_data())

In [ ]:
num_periods = 1
time_steps = 20
period_length = [pump_length] * num_periods
num_steps = [time_steps] * num_periods
timestep_mult = [1.5] * num_periods
pump_time_dis.perioddata = list(zip(period_length, num_steps, timestep_mult))

print(pump_time_dis.perioddata.get_data())

In [ ]:
def calculate_timesteps(perlen, nstp, tsmult):
    r"""
    Calculate time step lengths for a stress period using a geometric progression.

    Parameters
    ----------
    perlen : float
        Total length of the stress period.
    nstp : int
        Number of time steps in the stress period.
    tsmult : float
        Multiplier for the length of successive time steps. If 1.0, 
        time steps are of equal length.

    Returns
    -------
    steps : ndarray
        1D array of floats containing the length of each time step.

    Notes
    -----
    The length of the first time step (dt1) is calculated as:
    dt1 = perlen * (1 - tsmult) / (1 - tsmult^nstp)

    The rest of the timesteps are calculated as:
    dt1 * (tsmult ^ [0,1,2, ..., nstp]) -> 
                                        [dt1, dt1*tsmult, dt1*tsmult^2, ..., dt1*tsmult^nstp]
    """
    
    if tsmult == 1.0:
        return np.full(nstp, perlen / nstp)
    # "Vectorized" geometric progression
    dt1 = perlen * (1 - tsmult) / (1 - tsmult**nstp)
    return dt1 * (tsmult ** np.arange(nstp))

In [ ]:
help(calculate_timesteps)

In [ ]:
for p in range(num_periods):
    steps = calculate_timesteps(period_length[p], num_steps[p], timestep_mult[p])
    
    # Format all steps into a single string block
    steps_output = "\n".join([f"Timestep {i+1:2}: {val:.6f} days" for i, val in enumerate(steps)])
    
    print(f"--- Period {p+1} ---\n{steps_output}")
    print(f"Total Period Length: {sum(steps):.2f} days\n")

## Updating the Well Package (WEL)

The base model has an existing WEL package representing current pumping wells. We need to:

1. **Expand** the existing steady-state pumping data to all transient stress periods
2. **Append** the new production wells at the user-selected locations with the computed per-well pumping rate (negative = extraction)

The updated WEL data is applied to all stress periods, meaning all wells pump continuously at steady rates throughout the simulation.

In [ ]:
# --- Update the WEL Package ---
# Check for a WEL package, if it exists expand its data to all periods
if pump_gwf.get_package("wel"):
    wel = pump_gwf.get_package("wel")
    base_data = wel.stress_period_data.get_data(0)  # from steady-state
    
    # Safely convert to list, handling cases where base_data might be None
    if base_data is not None:
        base_data = base_data.tolist()
    else:
        base_data = []
        
    # Append each newly selected well with the evenly divided rate
    for well_cell in new_wells:
        # Negative value indicates extraction/pumping
        base_data.append((well_cell, -rate_per_well, None))
        
    # Apply to all stress periods
    wel_data = {per: base_data for per in range(num_periods)}
    wel.stress_period_data.set_data(wel_data)
    print(f"WEL package updated successfully with {len(new_wells)} new wells extracting {rate_per_well:.2f} m3/d each.")
else: 
    print("No WEL package found.")

## Setting Up Observation Points (OBS)

MODFLOW 6's OBS package writes simulated heads at specified cells to a CSV file at every time step. This gives us the **hydrograph** — the time series of head at each monitoring location.

We replace any existing OBS packages (from the base model calibration) with a new package containing only our user-selected observation points. Each observation name encodes its layer, row, and column for easy identification in the results file.

In [ ]:
# --- Update the OBS Package ---
custom_head_obs = []

out_file = "custom_head_obs.csv"

# Loop through the user-selected observation points
for obs_cell in obs_points:
    obs_layer, obs_row, obs_col = obs_cell
    
    # Create a 1-indexed name for the CSV headers
    obsname = f"obs_l{obs_layer + 1}_r{obs_row + 1}_c{obs_col + 1}"
    custom_head_obs.append((obsname, "HEAD", obs_cell))

# Base model has existing observations that are used during the calibration process.
# They are not relevant for this investigation, and we don't want to "muddy the waters"
# when we post-process and visualize our updated model.
# Remove old Roadcap/PEST/transect obs packages before adding the new custom obs package
for obs_pname in ["obs_0", "obs_1", "transect_head_obs"]:
    if pump_gwf.get_package(obs_pname):
        pump_gwf.remove_package(obs_pname)

# Create the new Observation package
flopy.mf6.ModflowUtlobs(
    pump_gwf,
    pname="obs_0",
    filename="gwf-iisg.obs",
    digits=10,
    print_input=True,
    continuous={
        out_file:custom_head_obs
    },
)

print(f"Replaced OBS package with {len(custom_head_obs)} custom head observations.")

## Adding the Storage Package (STO)

Transient simulations require a **Storage (STO) package** that tells MODFLOW how much water can be released from storage as heads decline:

| Parameter | Symbol | Typical Range | Description |
|---|---|---|---|
| Specific Storage | Ss | 10⁻⁶ – 10⁻⁴ m⁻¹ | Water released per unit aquifer volume per unit head decline (confined) |
| Specific Yield | Sy | 0.05 – 0.35 | Water released per unit area per unit head decline (unconfined) |

- `iconvert = 1` tells MODFLOW to automatically switch between confined and unconfined storage formulations based on simulated head
- `steady_state = False` and `transient = True` override the inherited steady-state flag

We first check whether a STO package already exists (it would if the model were already transient) before adding a new one.

In [ ]:
# Add storage
if pump_gwf.get_package("sto") is None:
    sto = flopy.mf6.ModflowGwfsto(
        pump_gwf,
        save_flows=True,
        iconvert=1,
        ss=1e-5,
        sy=0.15,
        steady_state=False,
        transient=True
    )

print(f"Specific Storage (Ss): {pump_gwf.sto.ss.array}")
print(f"Specific Yield (Sy): {pump_gwf.sto.sy.array}")

## Updating Initial Conditions (IC)

For a transient simulation, MODFLOW requires **initial head values** at every cell — representing conditions at time = 0.

We use the **steady-state heads from the base model** as our starting point. This is physically meaningful: the aquifer is assumed to be at its long-term equilibrium before pumping begins.

Cells with no-flow (HDRY) values greater than 1e20 are replaced with the layer bottom elevation to avoid numerical issues.

In [ ]:
# Update initial conditions

base_head_file = base_gwf.output.head()
base_times = base_head_file.get_times()
base_heads = base_head_file.get_data(totim=base_times[-1])


base_heads = np.where(base_heads > 1e20, layer_botm_elevations, base_heads)

print(base_heads[8])

ic = flopy.mf6.ModflowGwfic(pump_gwf, strt=base_heads)

print("Initial Conditions updated")

## Writing and Running the Simulation

`write_simulation()` writes all updated package files to the `new_demand_model/` directory, then `run_simulation()` launches the MODFLOW 6 executable.

MODFLOW will print a run log to the console. Look for **"Normal termination"** at the end, which confirms the simulation converged successfully. If you see convergence warnings:
- Reduce the time-step multiplier (try 1.2 instead of 1.5)
- Check that storage parameters are physically reasonable
- Verify that no wells are placed in dry cells

In [ ]:
pump_sim.write_simulation()
pump_sim.run_simulation()

## Post-Processing and Visualizing Results

Now that the simulation has run, we use the interactive `explore_model_results` dashboard to analyze the outputs. The dashboard provides:

### Plan-View Map
- **Drawdown mode**: contour lines show head decline from initial conditions, colored dark to light with the `magma` colormap
- **Head mode**: contour lines show the simulated potentiometric surface at the end of the simulation, colored with the `viridis` colormap
- **Red cells**: production wells (existing and new)
- **Green cells**: stream reaches

### Hydrograph Panel
- Each observation point's head or drawdown time series is plotted
- Click any observation marker on the map to **highlight** its hydrograph in orange
- The horizontal dashed line (head mode only) shows the cell bottom elevation — if the head drops below this, the cell may go dry

### Toggle
Use the **Drawdown / Head** toggle to switch between the two display modes. The map and hydrograph update reactively.

The function is defined first (with full docstring), then called in the cell below it.

In [ ]:
def explore_model_results(pump_gwf, new_wells, obs_points, obs_filename, selected_township, rows, cols, how_many_wells, placement_strategy, layer=8):
    """
    Builds an interactive dashboard to explore MODFLOW 6 results.
    
    Includes a toggleable plan view map (Heads vs Drawdown), dynamic legend, 
    value hover effects, top-level well plotting, and a reactive hydrograph panel.

    Parameters
    ----------
    pump_gwf : flopy.mf6.ModflowGwf
        The MODFLOW 6 groundwater flow model object for the pumping scenario.
    new_wells : list of tuple
        List of (layer, row, column) zero-based indices indicating the locations 
        of the newly proposed pumping wells.
    obs_points : list of tuple
        List of (layer, row, column) zero-based indices for the observation points.
    obs_filename : str
        The filename of the CSV containing the observation head results 
        (e.g., "custom_head_obs.csv").
    selected_township : str
        The name of the township being evaluated (used for the dashboard title).
    rows : array_like of int
        1D array or list of zero-based row indices defining the cells within 
        the selected township bounds.
    cols : array_like of int
        1D array or list of zero-based column indices defining the cells within 
        the selected township bounds.
    how_many_wells : int
        The total number of new wells added in this scenario.
    placement_strategy : str
        The spatial placement strategy used for the new wells (e.g., 'cluster' or 'diffuse').
    layer : int, optional
        The zero-based model layer index to extract and display 2D spatial contours 
        and well locations for. Default is 8.

    Returns
    -------
    None
        This function does not return a value. It renders an interactive `ipywidgets.VBox`
        dashboard directly in the Jupyter Notebook cell output.
    """
    # --- Data Extraction & Preparation ---
    results_df = pd.read_csv(Path(pump_gwf.model_ws) / obs_filename)
    
    obs_cols = [col for col in results_df.columns if col.lower() != "time"]
    
    ic_array = pump_gwf.ic.strt.array
    for obs in obs_cols:
        parts = obs.split("_")
        l = next(int(p.lower().replace("l", "")) - 1 for p in parts if p.lower().startswith("l"))
        r = next(int(p.lower().replace("r", "")) - 1 for p in parts if p.lower().startswith("r"))
        c = next(int(p.lower().replace("c", "")) - 1 for p in parts if p.lower().startswith("c"))
        
        initial_head = ic_array[l, r, c]
        
        results_df[f"{obs}_head"] = results_df[obs]
        results_df[f"{obs}_drawdown"] = initial_head - results_df[obs]
        
    head_file = pump_gwf.output.head()
    times = head_file.get_times()
    final_head_3d = head_file.get_data(totim=times[-1])
    
    final_head_2d = final_head_3d[layer]
    ic_2d = ic_array[layer]
    drawdown_2d = ic_2d - final_head_2d
    
    layer_botm_elevations = pump_gwf.modelgrid.botm
    
    # --- 2. Build the Base Cell GeoDataFrame ---
    polygons, hex_colors, head_vals, dd_vals = [], [], [], []
    has_new_wel = np.zeros((base_gwf.modelgrid.nrow, base_gwf.modelgrid.ncol), dtype=bool)
    for (wl, wr, wc) in new_wells:
        if wl == layer: has_new_wel[wr, wc] = True

    for r, c in zip(rows, cols):
        vertices = base_gwf.modelgrid.get_cell_vertices(r, c)
        polygons.append(Polygon(vertices))
        
        head_vals.append(final_head_2d[r, c])
        dd_vals.append(drawdown_2d[r, c])
        
        # We leave the new wells in the base layer for completeness, 
        # but they will also get a dedicated top layer later
        if has_new_wel[r, c] or has_wel[r, c]: 
            hex_colors.append('#FF0000') 
        elif has_sfr[r, c]: 
            hex_colors.append('#00FF00') 
        else: 
            hex_colors.append('#13294B') 
            
    cells_gdf = gpd.GeoDataFrame({
        'hex_color': hex_colors,
        'head': head_vals,
        'drawdown': dd_vals
    }, geometry=polygons, crs="EPSG:5070").to_crs(epsg=4326)
    cells_geojson = json.loads(cells_gdf.to_json())
    
    # --- 3. Build the Top-Level New Wells GeoDataFrame ---
    nw_polygons, nw_head_vals, nw_dd_vals = [], [], []
    for (wl, wr, wc) in new_wells:
        if wl == layer:
            vertices = base_gwf.modelgrid.get_cell_vertices(wr, wc)
            nw_polygons.append(Polygon(vertices))
            nw_head_vals.append(final_head_2d[wr, wc])
            nw_dd_vals.append(drawdown_2d[wr, wc])
            
    nw_layer = None
    if nw_polygons:
        nw_gdf = gpd.GeoDataFrame({
            'head': nw_head_vals,
            'drawdown': nw_dd_vals
        }, geometry=nw_polygons, crs="EPSG:5070").to_crs(epsg=4326)
        nw_json = json.loads(nw_gdf.to_json())
        
        def nw_style(feature):
            return {'fillColor': '#FF0000', 'color': '#13294B', 'weight': 1, 'fillOpacity': 0.8}
            
        nw_layer = GeoJSON(
            data=nw_json, 
            style_callback=nw_style, 
            hover_style={'fillOpacity': 1.0}, 
            name='Proposed Wells'
        )

    # --- Map Initialization & Layer Assembly ---
    tb = cells_gdf.total_bounds
    center_y, center_x = (tb[1] + tb[3]) / 2, (tb[0] + tb[2]) / 2
    m = Map(basemap=basemaps.Esri.WorldTopoMap, center=(center_y, center_x), zoom=11, layout=Layout(height='650px'))
    
    def cell_style(feature):
        color = feature['properties']['hex_color']
        opacity = 0.8 if color in ['#FF0000', '#00FF00'] else 0.05
        return {'fillColor': color, 'color': '#13294B', 'weight': 0.5, 'fillOpacity': opacity}
        
    cells_layer = GeoJSON(
        data=cells_geojson, 
        style_callback=cell_style, 
        hover_style={'fillOpacity': 0.4},
        name='Township Cells'
    )
    
    # Base cells added first
    m.add(cells_layer)
    
    # Contours added second
    contour_layer_group = LayerGroup()
    m.add(contour_layer_group)
    
    # New wells added third (plots OVER the contours)
    if nw_layer:
        m.add(nw_layer)
    
    # --- Contour & Legend Generation Engine ---
    legend_html = HTML()
    legend_control = WidgetControl(widget=legend_html, position='bottomright')
    m.add(legend_control)
    
    def generate_contours(mode):
        contour_layer_group.clear()
        
        min_r = max(0, rows.min() - 10)
        max_r = min(pump_gwf.modelgrid.nrow, rows.max() + 10)
        min_c = max(0, cols.min() - 10)
        max_c = min(pump_gwf.modelgrid.ncol, cols.max() + 10)
        
        target_array = drawdown_2d if mode == 'Drawdown' else final_head_2d
        subset = target_array[min_r:max_r, min_c:max_c].copy()
        
        subset[subset > 1e20] = np.nan
        subset[subset < -1e20] = np.nan
        
        X = pump_gwf.modelgrid.xcellcenters[min_r:max_r, min_c:max_c]
        Y = pump_gwf.modelgrid.ycellcenters[min_r:max_r, min_c:max_c]
        
        fig_d, ax_d = plt.subplots()
        cs = ax_d.contour(X, Y, subset, levels=15)
        
        lines, vals = [], []
        
        for i, level_segs in enumerate(cs.allsegs):
            val = cs.levels[i]
            for seg in level_segs:
                if len(seg) > 1:
                    lines.append(LineString(seg))
                    vals.append(val)
                    
        plt.close(fig_d)
        
        if len(lines) == 0: 
            legend_html.value = "<b>No Contours Found</b>"
            return
        
        c_gdf = gpd.GeoDataFrame({'value': vals}, geometry=lines, crs="EPSG:5070").to_crs(epsg=4326)
        
        norm = mcolors.Normalize(vmin=min(vals), vmax=max(vals))
        cmap = mpl.colormaps.get_cmap('magma' if mode == 'Drawdown' else 'viridis')
        c_gdf['color'] = c_gdf['value'].apply(lambda x: mcolors.to_hex(cmap(norm(x))))
        
        c_json = json.loads(c_gdf.to_json())
        
        def contour_style(feature):
            return {'color': feature['properties']['color'], 'weight': 3, 'opacity': 0.9}
            
        contour_layer_group.add(GeoJSON(data=c_json, style_callback=contour_style))
        
        unique_vals = sorted(list(set(vals)), reverse=True)
        legend_content = f"<div style='background-color: white; color: black; padding: 10px; border-radius: 5px; opacity: 0.9;'><b>{mode} Contours (m)</b><br>"
        for v in unique_vals:
            c = mcolors.to_hex(cmap(norm(v)))
            legend_content += f"<i style='background:{c}; width: 12px; height: 12px; display: inline-block; margin-right: 5px;'></i> {v:.2f}<br>"
        legend_content += "</div>"
        legend_html.value = legend_content

    # --- Interactive Hydrograph Panel ---
    plot_output = Output()
    state = {'mode': 'Drawdown', 'selected_obs': None}
    
    def render_plot():
        with plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(12, 4))
            mode = state['mode'].lower()
            
            time_col = next(col for col in results_df.columns if col.lower() == "time")
            
            for obs in obs_cols:
                y_vals = results_df[f"{obs}_{mode}"]
                
                parts = obs.split("_")
                l = next(int(p.lower().replace("l", "")) - 1 for p in parts if p.lower().startswith("l"))
                r = next(int(p.lower().replace("r", "")) - 1 for p in parts if p.lower().startswith("r"))
                c = next(int(p.lower().replace("c", "")) - 1 for p in parts if p.lower().startswith("c"))
                
                botm = layer_botm_elevations[l, r, c]
                
                if obs == state['selected_obs']:
                    ax.plot(results_df[time_col], y_vals, linewidth=3, color='#FF5F05', label=f"{obs} (Selected)", zorder=10)
                    if mode == 'head':
                        ax.axhline(botm, color='#FF5F05', linestyle='--', alpha=0.8, label="Cell Bottom")
                else:
                    ax.plot(results_df[time_col], y_vals, linewidth=1.5, color='#13294B', alpha=0.2)
                    
            ax.set_xlabel("Days into Pumping")
            ax.set_ylabel(f"{state['mode']} (m)")
            ax.set_title(f"Observation Node Results: {state['mode']}")
            if mode == 'drawdown': ax.invert_yaxis()
            ax.grid(True, alpha=0.3)
            
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                ax.legend(loc='upper right')
                
            plt.tight_layout()
            plt.show()

    # --- Event Listeners & UI Controls ---
    mode_toggle = ToggleButtons(
        options=['Drawdown', 'Head'],
        description='Display:',
        button_style='info'
    )
    
    hover_label = HTML(value="<b>Hovered Value:</b> <i>None</i>")
    info_label = HTML(value="<i>Click an observation point on the map to highlight its hydrograph.</i>")
    
    def on_toggle_change(change):
        if change['new']:
            state['mode'] = change['new']
            generate_contours(state['mode'])
            hover_label.value = f"<b>Hovered {state['mode']}:</b> <i>None</i>"
            render_plot()
            
    mode_toggle.observe(on_toggle_change, names='value')
    
    def on_cell_hover(event, feature, **kwargs):
        mode_key = state['mode'].lower()
        val = feature['properties'].get(mode_key, None)
        
        if val is None or val > 1e20 or val < -1e20:
            display_val = "Inactive"
        else:
            display_val = f"{val:.2f} m"
            
        hover_label.value = f"<b>Hovered {state['mode']}:</b> {display_val}"
        
    # Bind hover effect to both the base layer AND the top well layer
    cells_layer.on_hover(on_cell_hover)
    if nw_layer:
        nw_layer.on_hover(on_cell_hover)
    
    obs_group = LayerGroup()
    m.add(obs_group)
    
    marker_dict = {} 
    
    def update_marker_styles():
        for obs, marker in marker_dict.items():
            if obs == state['selected_obs']:
                marker.radius = 8
                marker.fill_color = '#FFFFFF'
                marker.color = '#FF5F05' 
                marker.weight = 4
            else:
                marker.radius = 6
                marker.fill_color = '#FF5F05'
                marker.color = '#13294B' 
                marker.weight = 2
    
    def make_obs_callback(obs_name):
        def callback(event, **kwargs):
            state['selected_obs'] = obs_name
            info_label.value = f"<b>Selected:</b> <span style='color:#FF5F05'>{obs_name}</span>"
            update_marker_styles()
            render_plot()
        return callback

    for obs in obs_cols:
        parts = obs.split("_")
        
        r = next(int(p.lower().replace("r", "")) - 1 for p in parts if p.lower().startswith("r"))
        c = next(int(p.lower().replace("c", "")) - 1 for p in parts if p.lower().startswith("c"))
        
        x = pump_gwf.modelgrid.xcellcenters[r, c]
        y = pump_gwf.modelgrid.ycellcenters[r, c]
        
        pt = gpd.GeoDataFrame(geometry=[gpd.points_from_xy([x], [y])[0]], crs="EPSG:5070").to_crs(epsg=4326)
        lat, lon = pt.geometry.y.iloc[0], pt.geometry.x.iloc[0]
        
        marker = CircleMarker(
            location=(lat, lon), radius=6, color='#13294B', fill_color='#FF5F05', fill_opacity=1.0, weight=2
        )
        marker.on_click(make_obs_callback(obs))
        obs_group.add(marker)
        marker_dict[obs] = marker 

    generate_contours(state['mode'])
    render_plot()
    
    # Assemble final dashboard with the new title
    dashboard_title = HTML(value=f"<h2 style='margin: 0 0 10px 0; color: #FF5F05;'>{selected_township} : {how_many_wells} new wells - {placement_strategy.title()} strategy</h2>")
    
    ui_controls = HBox([mode_toggle, hover_label, info_label], 
                       layout={'align_items': 'center', 'justify_content': 'space-between', 'margin': '0 0 10px 0'})
    
    dashboard = VBox([dashboard_title, ui_controls, m, plot_output])
    display(dashboard)


In [ ]:
# Execute the dashboard
explore_model_results(
    pump_gwf=pump_gwf,
    new_wells=new_wells, 
    obs_points=obs_points,
    obs_filename=out_file,
    selected_township=selected_township, 
    rows=rows, 
    cols=cols, 
    how_many_wells=how_many_wells,
    placement_strategy=placement_strategy,
    layer=8
)

## Summary

In this notebook, you:

1. ✅ Loaded the IISG base model and steady-state heads
2. ✅ Interactively selected a township and mapped its cells to the model grid
3. ✅ Used transmissivity to identify geologically favorable well locations
4. ✅ Placed new production wells interactively on an ipyleaflet map
5. ✅ Selected monitoring observation points to track the aquifer response
6. ✅ Copied the base model to a new workspace
7. ✅ Converted the simulation from steady-state to transient
8. ✅ Added storage parameters (Ss and Sy) required for transient simulation
9. ✅ Updated well, observation, and initial condition packages for the pumping scenario
10. ✅ Ran the modified simulation
11. ✅ Explored head and drawdown results in an interactive dashboard

### Suggested Extensions

- Adjust `how_many_wells` and compare **cluster vs. diffuse** placement strategies
- Increase `pumping_rate_mgd` to 20 MGD and observe the expanded drawdown cone
- Compare results from this notebook against `drought_impacts.ipynb` — how do the two stresses interact if both occur simultaneously?
- Extend `pump_length` to multiple years and add a recovery period (zero pumping)
- Export the final drawdown as a GIS layer and compare with population/land-use data